## Setup notes

- **Validation data.** `data/test.csv` is the Kaggle test file: it has only `id` and `text`, no labels, so no AUROC can be computed on it (the assignment also says to use *labeled validation data*). Validation here is therefore a **stratified 20% hold-out of `data/train.csv`** (`random_state=42`), the same split the team used in Part 4 (`fangfeidong/part4.py`). The model is trained on the remaining 8,000 rows and never sees the 2,000 validation rows.
- **Features.** 768-d embeddings from `embeddinggemma-2:270m` served by a local Ollama daemon (`ollama serve`), batched as in `lr.ipynb`. Embeddings are cached to `data/train_embeddinggemma2_270m.npy` so re-runs skip the ~6 min embedding step.
- **Model.** `SVC(kernel="poly", C=10.0)`. `probability=True` is added only to obtain P(AI) (Platt scaling) for the error analysis; it does not change the fitted decision boundary.

In [1]:
import os
import re
import time
import warnings

import numpy as np
import pandas as pd
import ollama

from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

# sklearn 1.9 deprecates SVC(probability=True); it still works and matches the team's setup
warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn")

RANDOM_SEED = 42
EMBED_MODEL = "embeddinggemma-2:270m"
THRESHOLD = 0.5  # P(AI) >= 0.5 -> predicted AI

df = pd.read_csv("./data/train.csv")
print(df.shape, df["label"].value_counts().to_dict())
print("test.csv columns:", list(pd.read_csv("./data/test.csv", nrows=1).columns), "(no label column)")

(10000, 3) {0: 5000, 1: 5000}
test.csv columns: ['id', 'text'] (no label column)


## Embed with EmbeddingGemma2:270M

In [2]:
def embed_texts(texts, model=EMBED_MODEL, batch_size=32):
    """Embed texts with Ollama in small batches (same approach as lr.ipynb)."""
    texts = np.asarray(texts, dtype=str).tolist()
    vectors = []
    for start in range(0, len(texts), batch_size):
        response = ollama.embed(model=model, input=texts[start : start + batch_size])
        vectors.extend(response["embeddings"])
    return np.asarray(vectors, dtype=np.float32)


CACHE_PATH = "./data/train_embeddinggemma2_270m.npy"

if os.path.exists(CACHE_PATH):
    X = np.load(CACHE_PATH)
    print(f"Loaded cached embeddings from {CACHE_PATH}")
else:
    start_time = time.time()
    X = embed_texts(df["text"])
    np.save(CACHE_PATH, X)
    print(f"Embedding time: {time.time() - start_time:.1f} seconds")

assert len(X) == len(df)
print(X.shape)

Loaded cached embeddings from ./data/train_embeddinggemma2_270m.npy
(10000, 768)


## Train / validation split and model

In [3]:
y = df["label"].to_numpy()
indices = np.arange(len(df))

train_idx, val_idx = train_test_split(
    indices, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)
X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]
print("train:", X_train.shape, "val:", X_val.shape)

model = SVC(kernel="poly", C=10.0, probability=True, random_state=RANDOM_SEED)

start_time = time.time()
model.fit(X_train, y_train)
print(f"Training time: {time.time() - start_time:.1f} seconds")

train: (8000, 768) val: (2000, 768)
Training time: 33.2 seconds
